# Series 2.5 — Long-Term AI Memory

**Why AI Fails? — Engineering Lab**

---

> Long-term memory is not a conversation archive.  
> It is a **compressed knowledge base**.

**Scenario:** 500 simulated conversations → extract memories → compress duplicates → retrieve only relevant facts for the prompt.

**Core lesson:** Conversation history grows forever. Long-term memory shouldn't.


## End-to-End Scenario

**What this lab does from start to finish**

A personal AI assistant has chatted with a user across **500 conversations** over months. Storing every message is impossible; storing every extracted fact without compression creates an unbounded memory store. This lab builds a **compressed long-term memory profile** and measures how much you can shrink it while still answering correctly.

### Step-by-step flow

1. **Load 500 simulated conversations** — `conversations.py` covers coding preferences, project history, tools, and temporary notes.
2. **Extract structured memories** — `memory_extractor.py` pulls category + key + value records (e.g. Language → Python, IDE → VS Code).
3. **Run four storage strategies:**
   - **No compression** — store every extracted memory (largest store and prompt)
   - **Deduplication** — same category+key → single memory
   - **Full compression** — dedup + consolidate related entries + update stale values + expire low-confidence/temporary items
   - **Compression + retrieval** — compressed store + inject only **top-K relevant** memories into the prompt
4. **Persist the profile** — `memory_store.py` writes a JSON user memory file.
5. **Answer benchmark questions** — `memory_retriever.py` finds relevant memories by intent + keyword; `prompts.py` injects them into the Gemini prompt.
6. **Evaluate retrieval** — `evaluator.py` measures whether the right memories were found and used.
7. **Compare all four strategies** — `benchmark.py` shows store size, prompt tokens, retrieval accuracy, and cost.
8. **Live demo cell** — run `python series-2.5/app.py --dry-run`.

### What you should observe

| Strategy | Memory store | Prompt injection | Tradeoff |
|----------|--------------|------------------|----------|
| No compression | Largest | All memories | Simple but unbounded |
| Dedup | Smaller | All unique memories | Removes duplicates only |
| Full compression | Much smaller | All compressed | Consolidates + expires |
| Compression + retrieval | Smallest store | Top-K only | Best production pattern |

**Takeaway:** Long-term memory is a **compressed knowledge base**, not a conversation archive. Extract → compress → retrieve only what's relevant.


## 1. The Problem

| Raw memory store | Compressed + retrieved memory |
|------------------|-------------------------------|
| Every extracted fact stored forever | Dedup, consolidate, expire obsolete entries |
| Full profile injected into every prompt | Top-K relevant memories only |
| Duplicates (FastAPI mentioned 50 times) | Single canonical preference |
| Obsolete facts (PyCharm → VS Code) | Latest wins, old entries dropped |

### Why this matters in production

- Personalization requires **remembering user preferences** across sessions
- Unbounded memory stores become slow, expensive, and contradictory
- The prompt should contain **relevant** memories — not the entire user profile

**Four strategies compared:**

```
raw        → no compression (baseline)
dedup      → same category+key → single memory
compressed → dedup + consolidate + update + expire
retrieval  → compressed store + top-K for prompt
```


## 2. What is Long-Term AI Memory?

**Long-term AI memory** stores **durable facts** extracted from many conversations and injects only **relevant** memories into each new request.

Unlike session summarization (Series 2.4), long-term memory **persists across sessions** and must be **compressed** to stay usable.

### Definition

```
Long-term memory = conversations → extract → compress → store → retrieve top-K → prompt
```

### Compression operations (`memory_compressor.py`)

| Operation | Example |
|-----------|---------|
| **Deduplication** | FastAPI mentioned 50× → one memory |
| **Consolidation** | FastAPI + Pydantic + SQLAlchemy → Python Backend Stack |
| **Updating** | PyCharm → VS Code (latest wins) |
| **Expiration** | Drop low-confidence / obsolete entries |

### What long-term memory is NOT

| Technique | Difference |
|-----------|------------|
| **Full chat log** | Memory stores **facts**, not transcripts |
| **RAG over docs** | Memory is **user-specific**, not document corpus |
| **Session summary** (2.4) | Summaries expire with the session; memory persists |

> **Enterprise principle:** Treat memory as a **knowledge base** with TTL, confidence scores, and retrieval — not a dump of every conversation.


## 3. Repository Layout

```
why-ai-fails/
├── common/
└── series-2.5/
    ├── app.py                 ← CLI benchmark entry
    ├── conversations.py       ← 500 simulated conversations
    ├── memory_extractor.py    ← Extract structured memories
    ├── memory_compressor.py   ← Dedup, consolidate, update, expire
    ├── memory_store.py        ← JSON profile storage
    ├── memory_retriever.py    ← Intent + keyword top-K
    ├── evaluator.py           ← Retrieval accuracy
    ├── benchmark.py
    ├── README.md
    └── Series_2.5_Long_Term_Memory.ipynb   ← This notebook
```


## 4. Python Files in This Lab

Every `.py` file under `series-2.5/`:

| File | What it does |
|------|--------------|
| **`app.py`** | CLI entry point. Runs the full pipeline across four memory strategies (raw → dedup → compressed → retrieval), prints benchmark. |
| **`conversations.py`** | Generates 500 simulated conversations with duplicates, changing preferences, TTL entries, and `EXPECTED_MEMORIES` for evaluation. |
| **`conversation_loader.py`** | `load_conversations()` — loads N conversations for processing. |
| **`memory_extractor.py`** | Regex-based extraction of structured memories (preferences, frameworks, security rules) from user turns via `EXTRACTION_RULES`. |
| **`memory_compressor.py`** | Four compression ops: `deduplicate()`, `consolidate()`, update (latest wins), and expire (drop low-confidence/obsolete). |
| **`memory_store.py`** | `MemoryStore` — JSON profile storage for compressed knowledge base. |
| **`memory_retriever.py`** | Intent + keyword retrieval — `detect_intents()`, ranks memories by question relevance, returns top-K. |
| **`evaluator.py`** | `retrieval_accuracy()` and personalization metrics vs expected memories. |
| **`prompts.py`** | `build_memory_prompt()` — injects only retrieved memories into prompt. |
| **`benchmark.py`** | Side-by-side strategy comparison printer. |


## 5. The Memory Pipeline

```
500 Conversations
    ↓
Memory Extractor
    ↓
Memory Compressor (dedup → consolidate → update → expire)
    ↓
Memory Store (JSON profile)
    ↓
Memory Retriever (top-K by question intent)
    ↓
Prompt Builder → Gemini
```

| Strategy | CLI | What it does |
|----------|-----|--------------|
| Raw | `--strategy raw` | Store every extracted memory |
| Dedup | `--strategy dedup` | Collapse duplicates |
| Compressed | `--strategy compressed` | Full compression pipeline |
| Retrieval | `--strategy retrieval` | Compressed + top-K in prompt |


## 6. Three Layers of Memory Engineering

### Layer 1 — Extract structured facts

Memories have `category`, `key`, `value`, `confidence`, and optional TTL — not free-form chat replay.

---

### Layer 2 — Compress the store

Without compression, 500 conversations produce thousands of redundant memories. Consolidation merges related stack items; expiration drops stale entries.

---

### Layer 3 — Retrieve + measure

| Metric | What it tells you |
|--------|-------------------|
| **Memory size** | Tokens in compressed store |
| **Prompt tokens** | Only top-K injected |
| **Retrieval accuracy** | Expected memories found / total |
| **Personalization** | Question-specific memory match |

| Mode | Flag | API key? |
|------|------|----------|
| **Dry-run** | `--dry-run` | No — **$0** |
| **Live** | (none) | Yes |


## 7. Execution Flow

```
Parse CLI (--strategy, --question-id, --conversations, --top-k)
    │
    └─ Load N conversations (default 500)
            → extract memories
            → apply strategy (raw / dedup / compressed / retrieval)
            → build prompt with memories
            → evaluate retrieval accuracy
            └─ print_benchmark()
```


## 8. How to Run

From the **repo root**:

```bash
pip install -r requirements.txt
cp .env.example .env   # optional
```

| Command | What it does | API key? |
|---------|--------------|----------|
| `python series-2.5/app.py --dry-run` | All four strategies | No |
| `python series-2.5/app.py --strategy retrieval --dry-run` | Best strategy | No |
| `python series-2.5/app.py --conversations 100 --dry-run` | Faster dev test | No |
| `python series-2.5/app.py` | Live Gemini | Yes |


In [ ]:
# Live demo cell — run the dry-run benchmark ($0, no API key needed)
# Execute this cell during your presentation

import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd()
if not (ROOT / "demo.py").exists() and (ROOT.parent / "demo.py").exists():
    ROOT = ROOT.parent

result = subprocess.run(
    [sys.executable, str(ROOT / "series-2.5/app.py"), "--dry-run"],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(result.stdout)
if result.stderr:
    print(result.stderr, file=sys.stderr)
print(f"\nExit code: {result.returncode}")


## 9. Key Code Snippets

### Deduplication (`memory_compressor.py`)

```python
def deduplicate(memories):
    # Same category+key → keep highest conversation_id / confidence
    best = {}
    for mem in memories:
        slot = (mem["category"], mem["key"])
        ...
    return list(best.values())
```

### Consolidation

```python
CONSOLIDATED_VALUE = "Python Backend Stack (FastAPI, Pydantic, SQLAlchemy)"
# FastAPI + Pydantic + SQLAlchemy → single backend_stack memory
```


## 10. Where Series 2.5 Fits

| Lab | Topic | Role |
|-----|-------|------|
| 2.4 | Conversation Summarization | Session-scoped memory |
| **2.5** | **Long-Term Memory** | **Build & compress** persistent user profile |
| 2.6 | Memory Retrieval | **Find** the right memory from 100k records |
| 2.7 | Model Routing | Select model tier per request |

---

## Takeaway

> **Store efficiently. Retrieve precisely. Never dump the full profile into every prompt.**

**Next lab:** [Series 2.6 — Memory Retrieval](../series-2.6/) — finding the right memory from a 100,000-record store.
